In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

a = spark.table("workspace.finance.bank_a_clean").withColumn("account", F.lit("bank_a"))
b = spark.table("workspace.finance.bank_b_clean").withColumn("account", F.lit("bank_b"))

all_txn = a.unionByName(b)     # matches columns by NAME, not position: safer
print("total rows:", all_txn.count())   # expect 299

total rows: 299


In [0]:
is_valid = (
    F.col("txn_date").isNotNull()
    & F.col("amount").isNotNull() & (F.col("amount") > 0)
    & F.col("direction").isNotNull()
)

valid = all_txn.filter(is_valid)

quarantine = (all_txn.filter(~is_valid)
    .withColumn("failed_rules", F.array_compact(F.array(
        F.when(F.col("txn_date").isNull(), F.lit("missing_or_bad_date")),
        F.when(F.col("amount").isNull() | (F.col("amount") <= 0), F.lit("missing_or_bad_amount")),
        F.when(F.col("direction").isNull(), F.lit("unknown_direction")),
    )))
    .withColumn("quarantined_at", F.current_timestamp()))

quarantine.write.mode("overwrite").saveAsTable("workspace.finance.quarantine_transactions")

print("valid:", valid.count(), " quarantined:", quarantine.count())   # expect 297 and 2
display(quarantine)

valid: 297  quarantined: 2


txn_date,description,bank_ref,amount,direction,source_file,account,failed_rules,quarantined_at
null,UPI/SWIGGY/swiggy@axisbank/Food order/352130348,716328933739,250.00,DEBIT,bank_a_statement_2026_07.csv,bank_a,List(missing_or_bad_date),2026-11-04T05:47:26.377Z
2026-05-15,POS 453212******1234 AMAZON PAY IN,null,null,DEBIT,bank_b_card_2026_05.csv,bank_b,List(missing_or_bad_amount),2026-11-04T05:47:26.377Z


In [0]:
def add_txn_id(df):
    # Everything that describes the transaction
    key = F.concat_ws("|",
        "account",
        F.col("txn_date").cast("string"),
        F.upper("description"),
        F.col("amount").cast("string"),
        "direction",
        F.coalesce("bank_ref", F.lit("")))

    # Number identical rows WITHIN the same file: 1, 2, 3...
    w = Window.partitionBy("source_file", "dedupe_key").orderBy(F.lit(1))

    return (df.withColumn("dedupe_key", key)
              .withColumn("occurrence", F.row_number().over(w))
              .withColumn("txn_id", F.sha2(F.concat_ws("|", "dedupe_key", "occurrence"), 256))
              .drop("dedupe_key"))

with_ids = add_txn_id(valid)
display(with_ids.select("txn_date", "description", "amount", "source_file", "occurrence", "txn_id"))

txn_date,description,amount,source_file,occurrence,txn_id
2026-04-01,NEFT-N2230054015-ACME TECHNOLOGIES PVT LTD-SALARY APR,95000.00,bank_a_statement_2026_04.csv,1,ccad6ef1298d65278f9b7a82833d0e3cf8078468dd5e55a3d4883dc36ab31675
2026-04-01,UPI/SWIGGY/swiggy@axisbank/Food order/136171878,194.00,bank_a_statement_2026_04.csv,1,2a3f7f5115d1c0ca0da5ab1499c92f27c1e3797c114d21d9ff71a1702ccd5507
2026-04-04,UPI/BLINKIT/blinkit@ybl/Quick order/317158893,472.21,bank_a_statement_2026_04.csv,1,24d077781e21e3b2685ad7b8324a8c3ab496a3b8c85f13dd9d9365df68998380
2026-04-05,IMPS-911856239313-RAJESH KUMAR-HOUSE RENT,18000.00,bank_a_statement_2026_04.csv,1,861e400a757106d9cc9b58a6cc3c9f85061418ba9d2f129384eb6046cb4afded
2026-04-05,UPI/BIGBASKET/bigbasket@icici/Groceries/590573056,1224.97,bank_a_statement_2026_04.csv,1,4e6b3ce157858715dc22e86d54c67d3a34c391808c04906157db81c46b02820f
2026-04-05,UPI/OLA CABS/olacabs@ybl/Ride/933251567,359.80,bank_a_statement_2026_04.csv,1,ad54489ea246929312c38538f776399e2f50412a4d49f42aebe94053ef64439d
2026-04-06,UPI/BIGBASKET/bigbasket@icici/Groceries/268697172,701.44,bank_a_statement_2026_04.csv,1,f0fec96934354a1d68bd5e497b7278ff74779897abaea0b48edccfe7bffbbfbf
2026-04-06,UPI/OLA CABS/olacabs@ybl/Ride/199223237,227.78,bank_a_statement_2026_04.csv,1,49828956dacadd711d526987a73c3893e52441b828bb9110adf91e3357dee601
2026-04-07,UPI/HP PETROL PUMP/hppay@icici/Fuel/390355792,1307.13,bank_a_statement_2026_04.csv,1,aeeb9f2058da2e24ea946880d68ff763e3a9777b7fd3f9bad11a2d9d401cac67
2026-04-08,UPI/BIGBASKET/bigbasket@icici/Groceries/516278088,510.32,bank_a_statement_2026_04.csv,1,b830f83bbabe8b834a20eb33a72b936b9beae07d01b0bb4ec8a4ea722032f465


In [0]:
# The duplicates: same txn_id from two different files
dupes = with_ids.groupBy("txn_id").agg(
    F.count("*").alias("copies"),
    F.collect_list("source_file").alias("files"),
    F.first("txn_date").alias("txn_date"),
    F.first("description").alias("description"))
display(dupes.filter("copies > 1").orderBy("txn_date"))

# Keep one copy per txn_id (from the earliest file)
w2 = Window.partitionBy("txn_id").orderBy("source_file")
deduped = (with_ids.withColumn("rn", F.row_number().over(w2))
                   .filter("rn = 1")
                   .drop("rn", "occurrence"))

print("after dedupe:", deduped.count())   # expect 272

txn_id,copies,files,txn_date,description
86c351c6c4316d5a362d0b725a61f720201ad6e686a2921b1bb4f5aba7689237,2,"List(bank_b_card_2026_04.csv, bank_b_card_2026_05.csv)",2026-05-01,POS 453212******1234 STARBUCKS COFFEE
872c4f8fec69f6fa91be935937a7c6123c39f4f0024ecacaadfbabdb6a4310da,2,"List(bank_a_statement_2026_04.csv, bank_a_statement_2026_05.csv)",2026-05-01,NEFT-N2307881371-ACME TECHNOLOGIES PVT LTD-SALARY MAY
4e84704889dc036df02db5bfe9e084b74bb322740dff08a602ea67cec83f9fa3,2,"List(bank_b_card_2026_04.csv, bank_b_card_2026_05.csv)",2026-05-03,NETFLIX.COM SUBSCRIPTION
79758703ad732ab1fa78fa2a8628dabdc04cc17c0b944952e6bb11cfb911b44a,2,"List(bank_a_statement_2026_04.csv, bank_a_statement_2026_05.csv)",2026-05-03,UPI/HP PETROL PUMP/hppay@icici/Fuel/662945045
ab5ea36bf92a234537e95e714e19c79d816db6cab6fdf7bc88afa334a463a79c,2,"List(bank_a_statement_2026_04.csv, bank_a_statement_2026_05.csv)",2026-05-03,UPI/UBER INDIA/uber@axis/Trip/173997741
3747fd45146eb8ada8e2d68b1806a22be7e93e9c3f09dd471abaaa773e27261f,2,"List(bank_b_card_2026_04.csv, bank_b_card_2026_05.csv)",2026-05-03,POS 453212******1234 AMAZON PAY IN
8aa1bf7b3f37e345f5f1a3cbd2002574d03cbec503b531b8074cc725da036a68,2,"List(bank_a_statement_2026_05.csv, bank_a_statement_2026_06.csv)",2026-06-01,UPI/CULT FIT/cultfit@razorpay/Membership/296755766
0dfd2a58691ee5b7f6020bd17265a16cbb87e3c80f4113402afac2c188ceea06,2,"List(bank_a_statement_2026_05.csv, bank_a_statement_2026_06.csv)",2026-06-01,UPI/SWIGGY/swiggy@axisbank/Food order/361691846
d4e85193c0ad2961a1be41f8d6c788a7ea9d88d31d98ec72e09f90f7ba571ae3,2,"List(bank_a_statement_2026_05.csv, bank_a_statement_2026_06.csv)",2026-06-01,NEFT-N3291700456-ACME TECHNOLOGIES PVT LTD-SALARY JUN
fdfdd9680a61c868c30151159a6f44d64aecfdca52bf4cf28668b38860b0a131,2,"List(bank_a_statement_2026_05.csv, bank_a_statement_2026_06.csv)",2026-06-02,UPI/SWIGGY/swiggy@axisbank/Food order/960420184


after dedupe: 272


In [0]:
display(deduped.filter("description LIKE '%STARBUCKS%'")
               .groupBy("txn_date", "amount").count()
               .filter("count > 1"))

txn_date,amount,count
2026-09-18,345.00,2
2026-06-18,345.00,2
2026-07-18,345.00,2
2026-08-18,345.00,2
2026-05-18,345.00,3
2026-04-18,345.00,2


In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.finance.silver_transactions (
  txn_id      STRING,
  account     STRING,
  txn_date    DATE,
  description STRING,
  bank_ref    STRING,
  amount      DECIMAL(12,2),
  direction   STRING,
  source_file STRING,
  loaded_at   TIMESTAMP
)
COMMENT 'Clean, de-duplicated transactions from all accounts';

In [0]:
(deduped.withColumn("loaded_at", F.current_timestamp())
        .createOrReplaceTempView("new_transactions"))

In [0]:
%sql
MERGE INTO workspace.finance.silver_transactions AS target
USING new_transactions AS source
ON target.txn_id = source.txn_id
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
272,0,0,272


In [0]:
reexport = (spark.table("workspace.finance.bank_b_clean")
    .filter("source_file = 'bank_b_card_2026_09.csv'")
    .withColumn("source_file", F.lit("bank_b_card_2026_09_REEXPORT.csv"))   # pretend it's a new file
    .withColumn("account", F.lit("bank_b")))

(add_txn_id(reexport.filter(is_valid))
    .drop("occurrence")
    .withColumn("loaded_at", F.current_timestamp())
    .createOrReplaceTempView("new_transactions"))

In [0]:
%sql
DESCRIBE HISTORY workspace.finance.silver_transactions;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-11-04T05:49:44.000Z,76004538558006,mmaverick254@gmail.com,MERGE,"Map(predicate -> [""(txn_id#11100 = txn_id#10883)""], clusterBy -> [], matchedPredicates -> [], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3289306133393965),edf7eb96-cc73-4c9b-8655-f8fb2dc26328,1104-054634-7fgiuwlw-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 17825, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 2149, materializeSourceTimeMs -> 21, numTargetRowsInserted -> 272, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 0, numTargetRowsUpdated -> 0, numOutputRows -> 272, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 272, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2064)",null,Databricks-Runtime/19.12.x-aarch64-photon-scala2.13
0,2026-11-04T05:49:21.000Z,76004538558006,mmaverick254@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> Clean, de-duplicated transactions from all accounts, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""io.unitycatalog.tableId"":""55a05673-83e3-410f-b922-1340ac635676"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-f702f057-c854-42b1-a33c-c6023eaacd87"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-07ce332e-4295-435f-8b7e-cdcbf4df2f1f""}, statsOnLoad -> false)",null,List(3289306133393965),54a7ed8e-4700-484c-a1b2-deca1b0c9d7e,1104-054634-7fgiuwlw-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/19.12.x-aarch64-photon-scala2.13
